In [2]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(tidyr)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
library(purrr)

# 1. Define folder path
folder_path <- "/share/crsp/lab/dalawson/tzhuravl/CellPlex_Tumors/Scanpy_Seurat/CODEX"

# 2. Find all .rds files
file_list <- list.files(path = folder_path, pattern = "\\.rds$", full.names = TRUE)

# 3. Stop if no files are found
if (length(file_list) == 0) {
  stop("No .rds files found in the specified folder. Please check your path.")
}

# 4. Load all objects into a named list
seurat_list <- setNames(lapply(file_list, readRDS), basename(file_list))

# 5. Safe merge logic based on the number of files found
if (length(seurat_list) == 1) {
  message("Only one RDS file found. No merge needed.")
  merged_seurat <- seurat_list[[1]]
} else {
  message(paste("Merging", length(seurat_list), "objects..."))
  merged_seurat <- merge(x = seurat_list[[1]], 
                         y = seurat_list[-1], 
                         add.cell.ids = names(seurat_list), 
                         project = "Merged_Project")
}

# 6. Optional (For Seurat v5): Join layers
if (length(seurat_list) > 1) {
  merged_seurat <- JoinLayers(merged_seurat)
}

Merging 9 objects...

Warning message:
“Key ‘fov_’ taken, using ‘fov2_’ instead”
Warning message:
“Key ‘fov_’ taken, using ‘fov3_’ instead”
Warning message:
“Key ‘fov_’ taken, using ‘fov4_’ instead”
Warning message:
“Key ‘fov_’ taken, using ‘fov5_’ instead”
Warning message:
“Key ‘fov_’ taken, using ‘fov6_’ instead”
Warning message:
“Key ‘fov_’ taken, using ‘fov7_’ instead”
Warning message:
“Key ‘fov_’ taken, using ‘fov8_’ instead”
Warning message:
“Key ‘fov_’ taken, using ‘fov9_’ instead”


In [4]:
merged_seurat

An object of class Seurat 
29 features across 140128 samples within 1 assay 
Active assay: Multiplex (29 features, 5 variable features)
 3 layers present: scale.data, data, counts
 9 spatial fields of view present: fov fov.2 fov.3 fov.4 fov.5 fov.6 fov.7 fov.8 fov.9

In [5]:
# export to Scanpy

# save metadata table:
merged_seurat$barcode <- colnames(merged_seurat)
write.csv(merged_seurat@meta.data, file='merged.codex.metadata.csv', quote=F,row.names=F)

# write expression counts matrix
library(Matrix)
counts_matrix <- GetAssayData(merged_seurat, assay='Multiplex', slot='counts')
writeMM(counts_matrix, file='merged.codex.counts.mtx')

# write gene names
write.table(
  data.frame('gene'=rownames(counts_matrix)),file='merged.codex.gene.names.csv',
  quote=F,row.names=F,col.names=F
) 


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”


NULL